# Chapter 7 - Cross-Validation in Finance

## Preparation

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold

from utils.sampling_bars import dollar_bar
from utils.labeling import get_daily_vol, symmetric_cusum_filter, get_vertical_bars, get_events, get_bins
from utils.cv import PurgedKFold

%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = 16,6

data_path = '../data/processed/clean_IVE_tickbidask.parq'
df = pd.read_parquet(data_path)

dollar_bars = dollar_bar(df)

In [2]:
dollar_bars.head()

,price,bid,ask,vol,dollar_vol,trade_count,price_high,price_low
date,,,,,,,,
2009-09-28 09:31:50,50.75,50.73,50.76,300.0,15225.00,1.0,50.75,50.75
2009-09-28 09:33:02,50.81,50.80,50.81,800.0,40648.00,2.0,50.81,50.81
2009-09-28 09:34:04,50.82,50.80,50.82,100.0,5082.00,1.0,50.82,50.82
2009-09-28 09:38:28,50.81,50.79,50.81,195.0,9907.95,1.0,50.81,50.81
2009-09-28 09:42:17,50.85,50.83,50.85,200.0,10170.00,2.0,50.85,50.85


In [3]:
close = dollar_bars["price"].sort_index()

## 1. Why is shuffling a dataset before conducting k-fold CV generally a bad idea in finance? What is the purpose of shuffling? Why does shuffling defeat the purpose of k-fold CV in financial datasets?

In the finance dataset, the order of the dataset matters, and they are not IID. Future data will depend on the history data. If we shuffle the data, we will leak the future information in the training dataset while training on both the history data, causing look-ahead bias. Besides, shuffling will destroy the temporal structure in the time series, such as serial correlation and momentum.

The purpose of shuffling is to prevent the model to learn the order information, which is typically noise in non-finance data. Besides, in typical datasets (like MNIST for image classification), the data are assumed to be IID, shuffling prevents a situation where one fold contains only samples of a certain class or feature range, which would produce unreliable CV scores.

## 2. Take a pair of matrices (X, y), representing observed features and labels. These could be one of the datasets derived from the exercises in Chapter 3.

In [4]:
# Trend-following strategy: moving average crossover
# 1. Compute short-term and long-term moving averages
short_window = 50
long_window = 200

short_ma = close.rolling(window=short_window, min_periods=1).mean()
long_ma = close.rolling(window=long_window, min_periods=1).mean()

# 2. Generate signals DataFrame with column name 'side'
signal = pd.DataFrame(index=close.index)
signal['side'] = 0
signal.loc[short_ma > long_ma, 'side'] = 1
signal.loc[short_ma < long_ma, 'side'] = -1

# Remove look-ahead bias by lagging the signal
signal['side'] = signal['side'].shift(1)

# 3. (Optional) Save or view the strategy signals
signal.to_csv("trend_follow_signals.csv")
signal.head()

daily_vol = get_daily_vol(close, span0=1000)
t_events = symmetric_cusum_filter(close, daily_vol)

vertical_bars = get_vertical_bars(close, t_events) # t1 in the text book

abs_return = close.pct_change().abs()

events = get_events(close, t_events=t_events, pt_sl=[1,2], t1=vertical_bars, num_threads=1, target=daily_vol, min_ret=0.01, side=signal['side'])

bins = get_bins(events, close) # meta labeling

In [5]:
values_counts_new = bins['bin'].value_counts()
values_counts_new

bin
0    775
1    332
Name: count, dtype: int64

In [6]:
# Create initial features

# Log Returns
dollar_bars['log_ret'] = np.log(dollar_bars['price']).diff()
dollar_bars['log_t1'] = dollar_bars['log_ret'].shift(1)
# prevent look-ahead bias
shifted_price = dollar_bars['price'].shift(1)

# Momentum
dollar_bars['mom2'] = shifted_price.pct_change(periods=2)
dollar_bars['mom3'] = shifted_price.pct_change(periods=3)

# Volatility
dollar_bars['volatility_50'] = dollar_bars['log_t1'].rolling(window=50, min_periods=50, center=False).std()
dollar_bars['volatility_15'] = dollar_bars['log_t1'].rolling(window=15, min_periods=15, center=False).std()

# Serial Correlation (Takes about 4 minutes)
window_autocorr = 50

dollar_bars['autocorr_1'] = dollar_bars['log_t1'].rolling(window=window_autocorr, min_periods=window_autocorr, center=False).apply(lambda x: x.autocorr(lag=1), raw=False)

# Get the various log -t returns
dollar_bars['log_t2'] = dollar_bars['log_ret'].shift(2)

In [7]:
# prepare train and test data
# Get features at event dates
feature_cols = ["volatility_50", "volatility_15", "autocorr_1", "log_t1", "log_t2", "mom2", "mom3"]
X = dollar_bars.loc[bins.index, feature_cols].copy()
X['side'] = bins['side']
y = bins['bin']

# Remove rows with missing feature values created by rolling/lag operations
xy = X.copy()
xy["y"] = y
xy = xy.dropna()

X_cv = xy.drop(columns=["y"])
y_cv = xy["y"]

In [8]:
print(X.shape)
print(X_cv.shape)

(1107, 8)
(1107, 8)


### a. Derive the performance from a 10-fold CV of an RF classifier on (X, y), without shuffling.

In [9]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
)

# 10-fold CV with no shuffling to preserve temporal ordering
kf = KFold(n_splits=10, shuffle=False)

acc_scores = cross_val_score(rf, X_cv, y_cv, cv=kf, scoring="accuracy", n_jobs=-1)
auc_scores = cross_val_score(rf, X_cv, y_cv, cv=kf, scoring="roc_auc", n_jobs=-1)

print(f"Accuracy (10-fold CV, no shuffle): {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}")
print(f"ROC AUC  (10-fold CV, no shuffle): {auc_scores.mean():.4f} +/- {auc_scores.std():.4f}")

pd.DataFrame({
    "fold": np.arange(1, 11),
    "accuracy": acc_scores,
    "roc_auc": auc_scores,
})

Accuracy (10-fold CV, no shuffle): 0.7002 +/- 0.0719
ROC AUC  (10-fold CV, no shuffle): 0.5632 +/- 0.0781


,fold,accuracy,roc_auc
0,1,0.639640,0.555062
1,2,0.702703,0.645556
2,3,0.729730,0.424330
3,4,0.855856,0.673341
4,5,0.648649,0.527778
5,6,0.585586,0.577441
6,7,0.657658,0.452885
7,8,0.754545,0.574297
8,9,0.681818,0.545279
9,10,0.745455,0.655923


### b. Derive the performance from a 10-fold CV of an RF on (X, y), with  shuffling.

In [10]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
)

# 10-fold CV with shuffling
kf = KFold(n_splits=10, shuffle=True)

acc_scores = cross_val_score(rf, X_cv, y_cv, cv=kf, scoring="accuracy", n_jobs=-1)
auc_scores = cross_val_score(rf, X_cv, y_cv, cv=kf, scoring="roc_auc", n_jobs=-1)

print(f"Accuracy (10-fold CV): {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}")
print(f"ROC AUC  (10-fold CV): {auc_scores.mean():.4f} +/- {auc_scores.std():.4f}")

pd.DataFrame({
    "fold": np.arange(1, 11),
    "accuracy": acc_scores,
    "roc_auc": auc_scores,
})

Accuracy (10-fold CV): 0.6965 +/- 0.0522
ROC AUC  (10-fold CV): 0.5636 +/- 0.0436


,fold,accuracy,roc_auc
0,1,0.756757,0.614682
1,2,0.729730,0.500741
2,3,0.657658,0.495370
3,4,0.720721,0.608473
4,5,0.693694,0.563218
5,6,0.594595,0.534677
6,7,0.720721,0.535284
7,8,0.727273,0.567493
8,9,0.745455,0.598394
9,10,0.618182,0.617500


### c. Why are both results so different?

Because we introduce look-ahead bias to the model with shuffling=True.

### d. How does shuffling leak information?

Shuffling will mix the future data and history data in a training set. For example, a training set of one fold may introduce the first year and the last year of the original dataset simultaneously.

## 3. Take the same pair of matrices (X, y) you used in exercise 2.

### a. Derive the performance from a 10-fold purged CV of an RF on (X, y),  with 1% embargo.

In [12]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1,
)

# 10-fold purged CV with 1% embargo
# Keep t1 aligned to X_cv/y_cv index (timestamps) to avoid index/type mismatches.
t1_cv = pd.Series(X_cv.index, index=X_cv.index)
purged_kf = PurgedKFold(n_splits=10, t1=t1_cv, pct_embargo=0.01)

acc_scores = cross_val_score(rf, X_cv, y_cv, cv=purged_kf, scoring="accuracy", n_jobs=-1)
auc_scores = cross_val_score(rf, X_cv, y_cv, cv=purged_kf, scoring="roc_auc", n_jobs=-1)

print(f"Accuracy (10-fold purged CV): {acc_scores.mean():.4f} +/- {acc_scores.std():.4f}")
print(f"ROC AUC  (10-fold purged CV): {auc_scores.mean():.4f} +/- {auc_scores.std():.4f}")

pd.DataFrame({
    "fold": np.arange(1, 11),
    "accuracy": acc_scores,
    "roc_auc": auc_scores,
})

Accuracy (10-fold purged CV): 0.6307 +/- 0.0769
ROC AUC  (10-fold purged CV): 0.5685 +/- 0.0638


,fold,accuracy,roc_auc
0,1,0.567568,0.607729
1,2,0.639640,0.647778
2,3,0.513514,0.486111
3,4,0.792793,0.663043
4,5,0.621622,0.514601
5,6,0.549550,0.560606
6,7,0.594595,0.462381
7,8,0.672727,0.565819
8,9,0.654545,0.556115
9,10,0.700000,0.621080


### b. Why is the performance lower?

Now there is few overlapped data among training and testing data, therefore the model will be less likely to cheat using the training data. By implementing purged K-Fold CV (and embargoing), we are explicitly removing this leaked information. 

### c. Why is this result more realistic?

Since we now have less overlapping data, the model will be less likely to overfit to the data, and there will be less leakage of the test data to the training data. The model is no longer "cheating" by accessing future or overlapping information, leading to a more honest and usually lower performance metric.

## 4. In this chapter we have focused on one reason why k-fold CV fails in financial applications, namely the fact that some information from the testing set leaks into the training set. Can you think of a second reason for CV's failure?

CV will construct the folds by splitting the dataset into K folds, and each fold will be a test set in some iteration. However, since a older fold maybe a test set, and we are using future data in the training set. It introduces look-ahead bias except the first iteration, where the last fold is used at the test set.

## 5. Suppose you try one thousand configurations of the same investment strategy, and perform a CV on each of them. Some results are guaranteed to look good, just by sheer luck. If you only publish those positive results, and hide the rest, your audience will not be able to deduce that these results are false positives, a statistical fluke. This phenomenon is called “selection bias.”

### a. Can you imagine one procedure to prevent this?

We can reduce the number of configurations in CV to prevent a higher likelihood of sheer luck.

Besides, we should deflate the Sharpe ratio by the number of trials carried out. This helps to account for the increased likelihood of finding an artificially high Sharpe ratio when many tests are performed.

We can also employ Combinatorial Purged Cross-Validation (CPCV) to addresses backtest overfitting by generating a distribution of performance metrics (e.g., Sharpe ratios) from a large number of paths. The objective is to make the variance of the backtest so small that the probability of a false discovery becomes negligible. If a researcher does not know the number and characteristics of the paths to be backtested in advance, their "overfitting efforts will be easily defeated".

### b. What if we split the dataset in three sets: training, validation, and testing?  The validation set is used to evaluate the trained parameters, and the testing is run only on the one configuration chosen in the validation phase. In what case does this procedure still fail?

The model may overfit to the validation dataset, since we are iteratively optimizing the validation set, and our hyperparameters are tuned for it. I am bound to find a configuration that performs exceptionally well on that specific validation set purely by chance, a "statistical fluke"

### c. What is the key to avoiding selection bias?

We can employ Purged K-Fold CV with Embargoing to avoid leakage of the test dataset on the train dataset by eliminating the overlap in both datasets. We can also record every backtest and estimate the probability of backtest overfitting (PBO) for the selected strategy. Besides, do not use backtest as the research tool. 